In [13]:
import os
import csv
import subprocess
import pandas as pd
from string import punctuation
import spacy
from wordfreq import zipf_frequency

spacy.prefer_gpu()

True

In [14]:
# Cell 2
def count_csv_elements_in_file(filepath):

    total_elements = 0

    try:
        with open(filepath, "r", encoding="utf-8") as file:
            csv_reader = csv.reader(file)
            for row in csv_reader:
                total_elements += len(row)
    except (UnicodeDecodeError, IOError):
        # Skip files that can't be read as UTF-8 text
        pass

    return total_elements

language = []
total_words = []

for path, subdirs, files in os.walk("raw-word-lists"):
    for name in files:
        # Only process .txt files; skip .DS_Store and README.md
        if not name.endswith(".txt"):
            continue
        filename = (os.path.join(path, name))
        language += [name.split(".")[0]]
        total_words += [count_csv_elements_in_file(filename)]

In [15]:
# Cell 3
pd.DataFrame({
    "language": language,
    "total_words": total_words
})

,language,total_words
0,Macedonian,3690
1,Dutch,165160
2,Catalan,3584
3,German,1707903
4,Lithuanian,82627
5,Romanian,272616
6,Russian,171552
7,Italian,661563
8,English,466434
9,Greek,35279


In [16]:
# Cell 4
spacy_models = {
    "Catalan": "ca_core_news_trf",
    "Croatian": "hr_core_news_lg",
    "Danish": "da_core_news_trf",
    "Dutch": "nl_core_news_lg",
    "English": "en_core_web_trf",
    "Finnish": "fi_core_news_lg",
    "French": "fr_dep_news_trf",
    "German": "de_dep_news_trf",
    "Greek": "el_core_news_lg",
    "Italian": "it_core_news_lg",
    "Lithuanian": "lt_core_news_lg",
    "Macedonian": "mk_core_news_lg",
    "Norwegian": "nb_core_news_lg",
    "Polish": "pl_core_news_lg",
    "Portuguese": "pt_core_news_lg",
    "Romanian": "ro_core_news_lg",
    "Russian": "ru_core_news_lg",
    "Slovenian": "sl_core_news_trf",
    "Spanish": "es_dep_news_trf",
    "Swedish": "sv_core_news_lg",
    "Ukrainian": "uk_core_news_trf"
}

In [18]:
# Cell 5
for model in spacy_models.values():
    result = subprocess.run(['python', '-m', 'spacy', 'download', f'{model}'],
                       capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

  Using cached ca_core_news_trf-3.8.0-py3-none-any.whl (457.1 MB)
✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_trf')

/Users/rickfisher/Development/jetbrains-yt-lang-learning-agent-project/language-learning-agent/.venv/lib/python3.15/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

  Using cached hr_core_news_lg-3.8.0-py3-none-any.whl (231.3 MB)
✔ Download and installation successful
You can now load the package via spacy.load('hr_core_news_lg')

/Users/rickfisher/Development/jetbrains-yt-lang-learning-agent-project/language-learning-agent/.venv/lib/python3.15/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

  Using cached da_core_news_trf-3.8.0-py3-no

In [ ]:
# Cell 6 (Dont need to run)
try:
    os.mkdir('data')
except FileExistsError:
    print('File or directory data already exists')

In [19]:
for language in spacy_models.keys():
    try:
        os.mkdir(f"data/{language}")
        print(f"Directory {language} created")
    except FileExistsError:
        print(f"Directory {language} already exists")

Directory Catalan already exists
Directory Croatian already exists
Directory Danish already exists
Directory Dutch already exists
Directory English already exists
Directory Finnish already exists
Directory French already exists
Directory German already exists
Directory Greek already exists
Directory Italian already exists
Directory Lithuanian already exists
Directory Macedonian already exists
Directory Norwegian already exists
Directory Polish already exists
Directory Portuguese already exists
Directory Romanian already exists
Directory Russian already exists
Directory Slovenian already exists
Directory Spanish already exists
Directory Swedish already exists
Directory Ukrainian already exists


In [13]:
def load_and_clean_word_list(language: str) -> pd.DataFrame:
    filepath = f"raw-word-lists/{language}/{language}.txt"
    if not os.path.exists(filepath):
        for f in os.listdir(f"raw-word-lists/{language}"):
            if f.endswith(".txt"):
                filepath = os.path.join(f"raw-word-lists/{language}", f)
                break
    with open(filepath, "r", encoding="utf-8") as f:
        word_list = f.read().split(",")

    word_df = pd.DataFrame({
        "word": word_list
    })

    word_df["word"] = word_df["word"].str.strip(punctuation)
    word_df = word_df[word_df["word"] != ""].dropna(subset=["word"])

    return word_df

In [10]:
load_and_clean_word_list("English")

,word
0,aardvark
1,aardvarks
2,aardwolf
3,aardwolves
4,Aaren
...,...
466429,Zwolle
466430,Zworykin
466431,ZZ
466432,zZt


In [8]:
nlp = spacy.load(spacy_models["English"], disable=["parser", "ner", "textcat"])

/Users/rickfisher/Development/jetbrains-yt-lang-learning-agent-project/language-learning-agent/.venv/lib/python3.15/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [14]:
# Cell 11
def add_lemma(df: pd.DataFrame,
              nlp,
              batch_size: int = 1000) -> pd.DataFrame:
    docs = nlp.pipe(df["word"].tolist(), batch_size=batch_size)
    lemmas = [doc[0].lemma_ if len(doc) > 0 else "" for doc in docs]
    df["lemma"] = pd.DataFrame(lemmas, index=df.index)
    return df

In [10]:
# Cell 12
add_lemma(
    load_and_clean_word_list("English")[:100], nlp
)

,word,lemma
0,aardvark,aardvark
1,aardvarks,aardvarks
2,aardwolf,aardwolf
3,aardwolves,aardwolf
4,Aaren,Aaren
...,...,...
95,Abama,Abama
96,abamp,abamp
97,abampere,abampere
98,abamperes,abamperes


In [15]:
# Cell 13
def add_word_frequencies(df: pd.DataFrame,
                         language: str) -> pd.DataFrame:
    language_group = spacy_models[language].split("_")[0]
    df["zipf_freq_lemma"] = [zipf_frequency(w, language_group) for w in df["lemma"]]
    return df

In [12]:
# Cell 14
add_word_frequencies(
    add_lemma(
        load_and_clean_word_list("English")[:100], nlp
), "English")

,word,lemma,zipf_freq_lemma
0,aardvark,aardvark,2.39
1,aardvarks,aardvarks,1.68
2,aardwolf,aardwolf,1.11
3,aardwolves,aardwolf,1.11
4,Aaren,Aaren,0.00
...,...,...,...
95,Abama,Abama,0.00
96,abamp,abamp,0.00
97,abampere,abampere,0.00
98,abamperes,abamperes,0.00


In [16]:
# Cell 15
def clean_up_and_export(df: pd.DataFrame, language: str) -> None:
    df = (
        df.loc[df.groupby("lemma", sort=False)["zipf_freq_lemma"].idxmax()]
        .reset_index(drop=True)
    )

    df = df[(df["zipf_freq_lemma"] > 0)]

    df.loc[:, "word_difficulty"] = pd.cut(
        df["zipf_freq_lemma"],
        bins = [-float("inf"), 2.0, 4.0, float("inf")],
        labels = ["advanced", "intermediate", "beginner"],
        include_lowest = True,
        right = True
    )

    df = df.drop(columns=["word", "zipf_freq_lemma"])
    df = df.rename(columns = {
        "lemma": "word"
    })

    os.makedirs(f"data/{language}", exist_ok=True)
    df.to_json(f"data/{language}/word-list-cleaned.json", orient="index")
    df.to_json(f"data/{language}/{language}.json", orient="index")
    os.makedirs(f"raw-word-lists/{language}", exist_ok=True)
    df.to_json(f"raw-word-lists/{language}/{language}.json", orient="index")

In [14]:
# Cell 16
clean_up_and_export(
    add_word_frequencies(
        add_lemma(
            load_and_clean_word_list("English")[:100], nlp),
        "English"),
    "English")

In [17]:
# Cell 17
def create_clean_word_list(language: str) -> None:
    nlp = spacy.load(spacy_models[language], disable=["parser", "ner", "textcat"])

    print("Load in dataset")
    lang_df = load_and_clean_word_list(language)

    print("Lemmatise words")
    lang_df = add_lemma(lang_df, nlp)

    print("Add the word frequencies")
    lang_df = add_word_frequencies(lang_df, language)

    print("Do the final clean ups and export to file")
    clean_up_and_export(lang_df, language)

    return None

In [17]:
# Cell 18
create_clean_word_list("Spanish")

Load in dataset
Lemmatise words
Add the word frequencies
Do the final clean ups and export to file


In [18]:
# Cell 19
import json

language_raw = []
total_words_raw = []

for path, subdirs, files in os.walk("raw-word-lists"):
    for name in files:
        if not name.endswith(".txt"):
            continue

        filename = (os.path.join(path, name))
        language_raw += [name.split(".")[0]]
        total_words_raw += [count_csv_elements_in_file(filename)]

raw_data = pd.DataFrame({
    "language": language_raw,
    "type": ["Raw"] * len(language_raw),
    "total_words_raw": total_words_raw,
})

In [21]:
# Cell 20

language_clean = []
total_words_clean = []

for path, subdirs, files in os.walk("data"):
    for name in files:

        if not name.endswith(".txt"):
            continue

        filename = (os.path.join(path, name))
        language_clean += [path.split("/")[1]]
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            total_words_clean += [len(data.keys())]

clean_data = pd.DataFrame({
    "language": language_raw,
    "type": ["Raw"] * len(language_raw),
    "total_words_raw": total_words_raw,
})

pd.concat([raw_data, clean_data])

NameError: name 'language_raw' is not defined

In [ ]:
# Cell 21
import gc

language_clean = []
total_words_clean = []

# a. Loop through all languages in folder 'raw-word-lists'
raw_word_lists_dir = "raw-word-lists"
languages = sorted([
    d for d in os.listdir(raw_word_lists_dir)
    if os.path.isdir(os.path.join(raw_word_lists_dir, d)) and not d.startswith(".")
])

for language in languages:
    lang_raw_dir = os.path.join(raw_word_lists_dir, language)
    json_in_raw = os.path.join(lang_raw_dir, f"{language}.json")
    json_in_data = os.path.join("data", language, f"{language}.json")
    json_cleaned_data = os.path.join("data", language, "word-list-cleaned.json")

    # If the language folder already has {language}.json file in it, assume that data processing completed and skip to the next language in the For Loop List.
    existing_json = None
    if os.path.exists(json_in_raw):
        existing_json = json_in_raw
    elif os.path.exists(json_in_data):
        existing_json = json_in_data
    elif os.path.exists(json_cleaned_data):
        existing_json = json_cleaned_data

    if existing_json:
        print(f"Skipping {language}: clean data already exists at {existing_json}")
        filename = existing_json
        language_clean += [language]
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            total_words_clean += [len(data.keys())]
        continue

    # c. Run data processing for languages without clean data
    if language in spacy_models:
        print(f"Processing clean word list for {language}...")
        create_clean_word_list(language)
        gc.collect()

        target_json = None
        if os.path.exists(json_in_raw):
            target_json = json_in_raw
        elif os.path.exists(json_in_data):
            target_json = json_in_data
        elif os.path.exists(json_cleaned_data):
            target_json = json_cleaned_data

        if target_json:
            filename = target_json
            language_clean += [language]
            with open(filename, "r", encoding="utf-8") as f:
                data = json.load(f)
                total_words_clean += [len(data.keys())]
            print(f"Completed {language}: {total_words_clean[-1]} words")
    else:
        print(f"Skipping {language}: no spaCy model configured")

# Cloned Cell #20 DataFrame creation and concat
clean_data = pd.DataFrame({
    "language": language_clean,
    "type": ["Clean"] * len(language_clean),
    "total_words_clean": total_words_clean,
})

pd.concat([raw_data, clean_data])

Skipping Catalan: clean data already exists at raw-word-lists/Catalan/Catalan.json
Skipping Croatian: clean data already exists at raw-word-lists/Croatian/Croatian.json
Processing clean word list for Danish...


/Users/rickfisher/Development/jetbrains-yt-lang-learning-agent-project/language-learning-agent/.venv/lib/python3.15/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Load in dataset
Lemmatise words
